# COMP64702 RAG Coursework
## Notebook 2: RAG Pipeline

Implements the full RAG pipeline:
- **Chunking** — sentence-boundary splitting with overlap; short/stub entries filtered
- **Embedding** — `all-MiniLM-L6-v2` dense embeddings with title prefix
- **Retrieval** — Hybrid BM25 + dense with Reciprocal Rank Fusion (RRF)
- **Generation** — Qwen2.5-0.5B-Instruct with adaptive prompting
- **Output** — saves `test_outputs.json` and `test_outputs_baseline.json`

**Files:**
- Input corpus: `Background_Corpus_Final_Combined.json`
- Input queries: `benchmark_input_only.json`
- Output: `test_outputs.json`

In [132]:
# Install dependencies (run once)
!pip install sentence-transformers transformers rank-bm25 accelerate -q

In [133]:
import json
import re
import numpy as np
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict
from collections import Counter

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from rank_bm25 import BM25Okapi
import torch

print("All imports OK")

All imports OK


In [134]:
DATA_DIR = Path(".")

corpus_path = DATA_DIR / "background_corpus.json"
query_path  = DATA_DIR / "benchmark_input_only.json"
output_path = DATA_DIR / "test_outputs.json"

with open(corpus_path, "r", encoding="utf-8") as f:
    corpus = json.load(f)

with open(query_path, "r", encoding="utf-8") as f:
    query_payload = json.load(f)

queries = query_payload["queries"]

print(f"Corpus: {len(corpus)} documents")
print(f"Queries: {len(queries)}")

Corpus: 204 documents
Queries: 100


In [135]:
@dataclass
class Chunk:
    doc_id: str
    title: str
    section: str
    text: str

@dataclass
class RetrievalResult:
    chunk: Chunk
    score: float

In [136]:
CHUNK_WORDS = 220
OVERLAP_WORDS = 40
MIN_ENTRY_CHARS = 300
MIN_CHUNK_WORDS = 25

def split_into_chunks(text: str, max_words=CHUNK_WORDS, overlap=OVERLAP_WORDS) -> List[str]:
    """Split text into overlapping sentence-boundary chunks."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks = []
    current = []
    current_len = 0

    for sent in sentences:
        words = sent.split()
        if current and current_len + len(words) > max_words:
            chunks.append(" ".join(current))

            # keep overlap
            overlap_sents = []
            overlap_len = 0
            for s in reversed(current):
                wlen = len(s.split())
                if overlap_len + wlen > overlap:
                    break
                overlap_sents.insert(0, s)
                overlap_len += wlen

            current = overlap_sents
            current_len = overlap_len

        current.append(sent)
        current_len += len(words)

    if current:
        chunks.append(" ".join(current))

    return chunks


def make_section_chunks(doc_id: str, title: str, section_name: str, text: str) -> List[Chunk]:
    text = text.strip()
    if len(text.split()) < MIN_CHUNK_WORDS:
        return []

    if len(text.split()) <= CHUNK_WORDS:
        return [Chunk(doc_id=doc_id, title=title, section=section_name, text=text)]

    parts = split_into_chunks(text)
    return [
        Chunk(
            doc_id=doc_id,
            title=title,
            section=f"{section_name} (Part {i+1})",
            text=part
        )
        for i, part in enumerate(parts)
        if len(part.split()) >= MIN_CHUNK_WORDS
    ]


def chunk_documents(corpus_docs: list) -> List[Chunk]:
    chunks = []

    for doc in corpus_docs:
        doc_id = doc.get("doc_id", "")
        title = doc.get("title", "")
        sections = doc.get("sections", [])
        full_text = doc.get("full_text", "").strip()

        # skip scaffold/meta docs
        if "scaffold" in doc_id or "scope" in doc_id:
            continue

        # skip extremely short docs
        if not sections and len(full_text) < MIN_ENTRY_CHARS:
            continue

        if sections:
            for sec in sections:
                heading = sec.get("heading", "Section")
                text = " ".join(sec.get("content", [])).strip()
                chunks.extend(make_section_chunks(doc_id, title, heading, text))
        else:
            chunks.extend(make_section_chunks(doc_id, title, "Full text", full_text))

    return chunks


chunks = chunk_documents(corpus)

print(f"Total chunks: {len(chunks)}")
print(f"Avg chunk length (words): {sum(len(c.text.split()) for c in chunks) // len(chunks)}")

chunk_id_map = {id(chunk): idx for idx, chunk in enumerate(chunks)}

wiki_chunks = [c for c in chunks if c.doc_id.startswith("wiki")]
print(f"Wiki chunks in index: {len(wiki_chunks)}")
print(f"Chunks under 30 words: {sum(1 for c in chunks if len(c.text.split()) < 30)}")

Total chunks: 446
Avg chunk length (words): 129
Wiki chunks in index: 116
Chunks under 30 words: 17


In [137]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded:", embedding_model.get_sentence_embedding_dimension(), "dims")

chunk_texts = [
    f"passage: {c.title}. {c.section}. {c.text}"
    for c in chunks
]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    batch_size=64,
    show_progress_bar=True
)

chunk_embeddings = chunk_embeddings.astype("float32")
chunk_embeddings_norm = chunk_embeddings / (np.linalg.norm(chunk_embeddings, axis=1, keepdims=True) + 1e-12)

print(f"Embeddings shape: {chunk_embeddings_norm.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7402.56it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: 384 dims


Batches: 100%|██████████| 7/7 [00:01<00:00,  4.44it/s]

Embeddings shape: (446, 384)


In [138]:
def tokenize(text: str) -> List[str]:
    text = text.lower()
    words = re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)?", text)
    bigrams = [f"{words[i]}_{words[i+1]}" for i in range(len(words)-1)]
    return words + bigrams

tokenized_chunks = [tokenize(t) for t in chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)

print("BM25 index built over", len(tokenized_chunks), "chunks")

BM25 index built over 446 chunks


In [139]:
DENSE_CANDIDATES = 20
BM25_CANDIDATES = 20

def dense_retrieve(query: str, top_k: int = DENSE_CANDIDATES) -> List[RetrievalResult]:
    q_text = f"query: {query}"
    q_emb = embedding_model.encode([q_text], convert_to_numpy=True)[0].astype("float32")
    q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-12)

    sims = chunk_embeddings_norm @ q_emb
    top_idx = np.argsort(sims)[::-1][:top_k]

    return [RetrievalResult(chunk=chunks[i], score=float(sims[i])) for i in top_idx]


def bm25_retrieve(query: str, top_k: int = BM25_CANDIDATES) -> List[RetrievalResult]:
    scores = bm25.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:top_k]

    return [RetrievalResult(chunk=chunks[i], score=float(scores[i])) for i in top_idx]

In [140]:
RRF_K = 60
TOP_K = 5
MAX_PER_DOC = 3

def hybrid_retrieve(query: str, top_k: int = TOP_K) -> List[RetrievalResult]:
    dense_res = dense_retrieve(query, top_k=20)
    bm25_res  = bm25_retrieve(query, top_k=20)

    score_map: Dict[str, dict] = {}

    for rank, res in enumerate(dense_res, 1):
        key = f"{res.chunk.doc_id}||{res.chunk.section}||{hash(res.chunk.text)}"
        score_map.setdefault(key, {"chunk": res.chunk, "score": 0.0})
        score_map[key]["score"] += 1.0 / (RRF_K + rank)

    for rank, res in enumerate(bm25_res, 1):
        key = f"{res.chunk.doc_id}||{res.chunk.section}||{hash(res.chunk.text)}"
        score_map.setdefault(key, {"chunk": res.chunk, "score": 0.0})
        score_map[key]["score"] += 1.0 / (RRF_K + rank)

    merged = sorted(score_map.values(), key=lambda x: x["score"], reverse=True)

    selected = []
    doc_counts = {}

    for item in merged:
        chunk = item["chunk"]
        did = chunk.doc_id

        if doc_counts.get(did, 0) >= MAX_PER_DOC:
            continue

        selected.append(RetrievalResult(chunk=chunk, score=item["score"]))
        doc_counts[did] = doc_counts.get(did, 0) + 1

        if len(selected) == top_k:
            break

    return selected

def baseline_retrieve(query: str, top_k: int = TOP_K) -> List[RetrievalResult]:
    return dense_retrieve(query, top_k=top_k)

In [141]:
test_query = "What is biryani and which region is it originally associated with?"
test_results = hybrid_retrieve(test_query)

print(f"Top {len(test_results)} chunks retrieved:\n")
for i, r in enumerate(test_results, 1):
    print(f"{i}. [{r.chunk.doc_id} | {r.chunk.section}] score={r.score:.4f}")
    print(f"   {r.chunk.text[:160]}...")
    print()

Top 5 chunks retrieved:

1. [wiki_biryani | Introduction] score=0.0323
   Biryani is a rice dish made with rice, spices, and usually meat, although vegetarian versions also exist. It is one of the best-known dishes associated with Sou...

2. [india_dish_biryani | Full text (Part 2)] score=0.0313
   Regional varieties include Hyderabadi biryani (made with goat meat and saffron, considered a culinary emblem of the city), Lucknawi Awadhi biryani, Kolkata biry...

3. [bangladesh_dish_kacchi_biryani | Full text (Part 2)] score=0.0304
   The handi is sealed with dough (the dum technique) and cooked over very low heat for 45–60 minutes. Kacchi biryani is associated with Dhaka's Old City (particul...

4. [india_regional_hyderabadi_cuisine | Full text (Part 1)] score=0.0258
   Hyderabadi cuisine (also called Deccani cuisine) is the culinary tradition of the city of Hyderabad and its historic surrounding region in the Deccan plateau — ...

5. [crossregional_dish_biryani_variants | Full text (Par

In [142]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()
print(f"Model loaded on: {next(model.parameters()).device}")

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 203.23it/s]


Model loaded on: mps:0


In [ ]:
def split_sentences(text: str) -> List[str]:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if len(s.strip()) > 20]

def sentence_overlap_score(query: str, sentence: str) -> float:
    q_tokens = set(re.findall(r"[a-z0-9]+", query.lower()))
    s_tokens = set(re.findall(r"[a-z0-9]+", sentence.lower()))
    if not q_tokens:
        return 0.0

    overlap = len(q_tokens & s_tokens) / len(q_tokens)

    q_lower = query.lower()
    s_lower = sentence.lower()

    if any(w in q_lower for w in ["ingredient", "ingredients", "include", "contain"]):
        if any(w in s_lower for w in ["include", "includes", "contains", "made with", "ingredients"]):
            overlap += 0.15

    if any(w in q_lower for w in ["what is", "what are"]):
        if any(w in s_lower for w in [" is ", " are ", "refers to", "dish made from"]):
            overlap += 0.10

    if "served with" in q_lower and "served with" in s_lower:
        overlap += 0.20

    if "introduced" in q_lower and "introduced" in s_lower:
        overlap += 0.20

    return overlap

def extract_relevant_sentences(query: str, retrieved: List[RetrievalResult], max_sentences: int = 8) -> List[str]:
    scored = []

    for r in retrieved:
        for sent in split_sentences(r.chunk.text):
            score = sentence_overlap_score(query, sent)
            scored.append((score, r.chunk.title, r.chunk.section, sent))

    scored = sorted(scored, key=lambda x: x[0], reverse=True)

    selected = []
    seen = set()

    for score, title, section, sent in scored:
        key = sent.strip().lower()
        if key in seen:
            continue
        seen.add(key)
        selected.append(f"[Source: {title} | {section}] {sent}")
        if len(selected) == max_sentences:
            break

    return selected

In [ ]:
MAX_CONTEXT_CHARS = 1200

def build_prompt(query: str, retrieved: List[RetrievalResult]) -> List[dict]:
    evidence_sentences = extract_relevant_sentences(query, retrieved, max_sentences=8)
    evidence = "\n".join(evidence_sentences)

    q_lower = query.lower()

    if any(w in q_lower for w in ["ingredient", "ingredients", "include", "contain"]):
        answer_instruction = (
            "Answer with only the ingredients explicitly mentioned in the evidence. "
            "Do not add any ingredient that is not stated. "
            "Write 1 to 2 complete sentences."
        )
    elif "served with" in q_lower:
        answer_instruction = (
            "Answer only with what the dish is served with, using the evidence. "
            "Write 1 short sentence."
        )
    elif any(w in q_lower for w in ["what is", "what are"]):
        answer_instruction = (
            "Give a short direct definition using only the evidence. "
            "Do not add preparation details unless asked. "
            "Write 1 to 2 complete sentences."
        )
    elif "introduced" in q_lower:
        answer_instruction = (
            "Answer only with the techniques or items explicitly stated as introduced in the evidence. "
            "Do not add unrelated historical details. "
            "Write 1 to 2 complete sentences."
        )
    else:
        answer_instruction = (
            "Answer using only the evidence below. "
            "Be accurate and specific. "
            "Write 1 to 2 complete sentences."
        )

    return [
        {
            "role": "system",
            "content": (
                "You are a South Asian cuisine assistant. "
                "Answer only from the evidence provided below. "
                "Do not guess. "
                "Do not add outside knowledge. "
                "If the evidence is insufficient, say so clearly."
            )
        },
        {
            "role": "user",
            "content": (
                f"Evidence:\n{evidence}\n\n"
                f"Question: {query}\n\n"
                f"{answer_instruction}"
            )
        }
    ]

In [ ]:
def postprocess(text: str) -> str:
    text = text.strip()

    for prefix in ["Answer:", "Response:", "A:"]:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()

    text = re.sub(r"\s+", " ", text).strip()

    # remove trailing incomplete fragment only if clearly cut off
    if text and text[-1] not in ".!?":
        parts = re.split(r'(?<=[.!?])\s+', text)
        if len(parts) > 1:
            text = " ".join(parts[:-1]).strip()

    return text


def generate(messages: list) -> str:
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            temperature=None,
            top_p=None,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )

    new_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return postprocess(tokenizer.decode(new_ids, skip_special_tokens=True))

In [145]:
sample_query = queries[0]["query"]
sample_results = hybrid_retrieve(sample_query)
sample_messages = build_prompt(sample_query, sample_results)
sample_answer = generate(sample_messages)

print("QUERY: ", sample_query)
print("ANSWER:", sample_answer)
print()
print("Retrieved docs:")
for r in sample_results:
    print(f"  - {r.chunk.doc_id} | {r.chunk.section}")

QUERY:  What is biryani and which culinary tradition is it most closely associated with?
ANSWER: Biryani is a rice dish made with rice, spices, and usually meat, but vegetarian versions also exist. It is one of the best-known dishes associated with South Asian cuisine.

Retrieved docs:
  - wiki_biryani | Introduction
  - india_dish_biryani | Full text (Part 2)
  - india_regional_hyderabadi_cuisine | Full text (Part 1)
  - bangladesh_dish_kacchi_biryani | Full text (Part 2)
  - afghanistan_dish_bolani | Full text (Part 2)


In [146]:
results = []

for i, item in enumerate(queries):
    query_id = item["query_id"]
    query = item["query"]

    retrieved = hybrid_retrieve(query, top_k=TOP_K)
    messages = build_prompt(query, retrieved)
    response = generate(messages)

    results.append({
        "query_id": query_id,
        "query": query,
        "response": response,
        "retrieved_context": [
            {
                "doc_id": r.chunk.doc_id,
                "text": r.chunk.text
            }
            for r in retrieved
        ]
    })

    if (i + 1) % 10 == 0:
        print(f"{i+1}/{len(queries)} done")

print(f"\nGenerated {len(results)} answers")

10/100 done
20/100 done
30/100 done
40/100 done
50/100 done
60/100 done
70/100 done
80/100 done
90/100 done
100/100 done

Generated 100 answers


In [147]:
with open(output_path, "w", encoding="utf-8") as f:
    json.dump({"results": results}, f, indent=2, ensure_ascii=False)

print(f"Saved {len(results)} results to {output_path}")

Saved 100 results to test_outputs.json


In [148]:
baseline_results = []

for i, item in enumerate(queries):
    query_id = item["query_id"]
    query = item["query"]

    retrieved = baseline_retrieve(query, top_k=TOP_K)
    messages = build_prompt(query, retrieved)
    response = generate(messages)

    baseline_results.append({
        "query_id": query_id,
        "query": query,
        "response": response,
        "retrieved_context": [
            {"doc_id": r.chunk.doc_id, "text": r.chunk.text}
            for r in retrieved
        ]
    })

    if (i + 1) % 10 == 0:
        print(f"{i+1}/{len(queries)} done")

baseline_path = DATA_DIR / "test_outputs_baseline.json"
with open(baseline_path, "w", encoding="utf-8") as f:
    json.dump({"results": baseline_results}, f, indent=2, ensure_ascii=False)

print(f"Baseline saved to {baseline_path}")

10/100 done
20/100 done
30/100 done
40/100 done
50/100 done
60/100 done
70/100 done
80/100 done
90/100 done
100/100 done
Baseline saved to test_outputs_baseline.json


In [153]:
demo_query = input("Enter your question: ")
demo_retrieved = hybrid_retrieve(demo_query)
demo_messages = build_prompt(demo_query, demo_retrieved)
demo_answer = generate(demo_messages)

print("\n── ANSWER ─────────────────────────────────────────────────")
print(demo_answer)

print("\n── RETRIEVED CONTEXT ──────────────────────────────────────")
for i, r in enumerate(demo_retrieved, 1):
    print(f"\n{i}. [{r.chunk.doc_id} | {r.chunk.section}] (score: {r.score:.5f})")
    print(r.chunk.text[:300])


── ANSWER ─────────────────────────────────────────────────
Ghevar is traditionally served with rabri, a thick, reduced, sweetened milk dessert flavored with cardamom and saffron.

── RETRIEVED CONTEXT ──────────────────────────────────────

1. [india_dish_ghevar | Full text] (score: 0.03279)
Ghevar is a traditional Rajasthani disc-shaped sweet made from a batter of refined wheat flour (maida), ghee, and water, deep-fried in ghee in a special cylindrical mould to produce a porous, lattice-like golden cake that is then soaked in sugar syrup. The characteristic honeycomb texture comes from

2. [india_ingredient_ghee | Full text (Part 2)] (score: 0.01613)
It is considered a sacred and auspicious food in Hindu tradition. Ghee contains negligible amounts of lactose and casein, making it generally tolerable for those with mild dairy sensitivities. It is a rich source of fat-soluble vitamins A, D, E, and K, though its high saturated fat content means mod

3. [wiki_sri_lankan_cuisine | Kiriba